In [ ]:
import pandas as pd
df = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)
print(df.columns)
print(df.info())
print(df['OnlineSecurity'].unique())
print(df['TechSupport'].unique())
print(df['Partner'].unique())
print(df['Dependents'].unique())
print(df['PhoneService'].unique())
print(df['MultipleLines'].unique())
print(df['OnlineBackup'].unique())
print(df['DeviceProtection'].unique())
print(df['StreamingTV'].unique())
print(df['StreamingMovies'].unique())
print(df['PaperlessBilling'].unique())
print(df['Churn'].unique())

In [ ]:
pip install catboost

In [ ]:
# ====== IMPORTS ======
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ====== LOAD DATA ======
df = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# ====== PREPROCESSING ======
# TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(" ",0))

# Replace 'No internet service' or 'No phone service' with 'No'
replace_cols = ['MultipleLines','OnlineSecurity','OnlineBackup','DeviceProtection',
                'TechSupport','StreamingTV','StreamingMovies']
for col in replace_cols:
    df[col] = df[col].replace({'No internet service':'No','No phone service':'No'})

# Binary mapping
binary_cols = ['Partner','Dependents','PhoneService','PaperlessBilling','Churn']
for col in binary_cols:
    df[col] = df[col].map({'Yes':1,'No':0})

# Feature engineering
df['NumServices'] = ((df['PhoneService']==1).astype(int) +
                     (df['MultipleLines']=='Yes').astype(int) +
                     (df['OnlineSecurity']=='Yes').astype(int) +
                     (df['OnlineBackup']=='Yes').astype(int) +
                     (df['DeviceProtection']=='Yes').astype(int) +
                     (df['TechSupport']=='Yes').astype(int) +
                     (df['StreamingTV']=='Yes').astype(int) +
                     (df['StreamingMovies']=='Yes').astype(int))

# Tenure group
df['TenureGroup'] = pd.cut(df['tenure'], bins=[0,12,24,48,60,72], labels=[1,2,3,4,5])
df['TenureGroup'] = df['TenureGroup'].astype(str)

# CatBoost categorical features
cat_features = ['gender','MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
                'DeviceProtection','TechSupport','StreamingTV','StreamingMovies',
                'Contract','PaymentMethod','TenureGroup']

# ====== TRAIN-TEST SPLIT ======
X = df.drop(['customerID','Churn'], axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

train_pool = Pool(data=X_train, label=y_train, cat_features=cat_features)
test_pool = Pool(data=X_test, label=y_test, cat_features=cat_features)

# ====== TRAIN CATBOOST ======
cb_model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.1,
    class_weights=[1,2],  # Handle class imbalance
    random_seed=42,
    verbose=100
)
cb_model.fit(train_pool, eval_set=test_pool)

# ====== EVALUATION ======
y_pred = cb_model.predict(X_test)
y_prob = cb_model.predict_proba(X_test)[:,1]

print("\n📊 Model Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

# Feature importance plot
feat_imp = cb_model.get_feature_importance(train_pool)
plt.figure(figsize=(10,6))
plt.barh(X_train.columns, feat_imp)
plt.title("Feature Importance")
plt.show()

# ====== CUSTOMER SEGMENTATION DASHBOARD ======
X_test_copy = X_test.copy()
X_test_copy['ChurnProb'] = y_prob
X_test_copy['RiskCategory'] = pd.cut(y_prob, bins=[0,0.4,0.7,1], labels=['Low','Medium','High'])
print("\nSample Customer Segmentation (first 5 rows):")
print(X_test_copy[['ChurnProb','RiskCategory']].head())

# ====== SIMPLE UI ======
def get_input(prompt, options=None):
    while True:
        val = input(f"{prompt}: ")
        if options and val not in options:
            print(f"Enter valid option: {options}")
        else:
            return val

def predict_churn():
    print("\n--- Enter Customer Details ---")
    gender = get_input("Gender (Male/Female)", ['Male','Female'])
    senior = get_input("Senior Citizen? (Yes/No)", ['Yes','No'])
    partner = get_input("Partner? (Yes/No)", ['Yes','No'])
    dependents = get_input("Dependents? (Yes/No)", ['Yes','No'])
    tenure = int(get_input("Tenure (in months)"))
    phone = get_input("Phone Service? (Yes/No)", ['Yes','No'])
    multiple = get_input("Multiple Lines? (Yes/No)", ['Yes','No'])
    internet = get_input("Internet Service (DSL/Fiber/No)", ['DSL','Fiber','No'])
    contract = get_input("Contract Type (Month-to-month/One year/Two year)", ['Month-to-month','One year','Two year'])
    paperless = get_input("Paperless Billing? (Yes/No)", ['Yes','No'])
    payment = get_input("Payment Method (Electronic check/Mailed check/Bank transfer/Credit card)",
                        ['Electronic check','Mailed check','Bank transfer','Credit card'])
    monthly = float(get_input("Monthly Charges"))
    total = float(get_input("Total Charges"))

    # Services
    online_sec = get_input("Online Security? (Yes/No)", ['Yes','No'])
    online_bk = get_input("Online Backup? (Yes/No)", ['Yes','No'])
    device_prot = get_input("Device Protection? (Yes/No)", ['Yes','No'])
    tech_supp = get_input("Tech Support? (Yes/No)", ['Yes','No'])
    streaming_tv = get_input("Streaming TV? (Yes/No)", ['Yes','No'])
    streaming_movies = get_input("Streaming Movies? (Yes/No)", ['Yes','No'])

    num_services = sum([phone=='Yes', multiple=='Yes', online_sec=='Yes', online_bk=='Yes',
                        device_prot=='Yes', tech_supp=='Yes', streaming_tv=='Yes', streaming_movies=='Yes'])

    # Tenure group
    if tenure <=12: tenure_group='1'
    elif tenure<=24: tenure_group='2'
    elif tenure<=48: tenure_group='3'
    elif tenure<=60: tenure_group='4'
    else: tenure_group='5'

    input_df = pd.DataFrame([{
        'gender':gender,
        'SeniorCitizen':1 if senior=='Yes' else 0,
        'Partner':1 if partner=='Yes' else 0,
        'Dependents':1 if dependents=='Yes' else 0,
        'tenure':tenure,
        'PhoneService':1 if phone=='Yes' else 0,
        'MultipleLines':multiple,
        'InternetService':internet,
        'OnlineSecurity':online_sec,
        'OnlineBackup':online_bk,
        'DeviceProtection':device_prot,
        'TechSupport':tech_supp,
        'StreamingTV':streaming_tv,
        'StreamingMovies':streaming_movies,
        'Contract':contract,
        'PaperlessBilling':1 if paperless=='Yes' else 0,
        'PaymentMethod':payment,
        'MonthlyCharges':monthly,
        'TotalCharges':total,
        'NumServices':num_services,
        'TenureGroup':tenure_group
    }])

    pred_prob = cb_model.predict_proba(Pool(input_df, cat_features=cat_features))[0][1]
    pred = cb_model.predict(Pool(input_df, cat_features=cat_features))[0]

    # Risk category
    if pred_prob > 0.7: risk="High"
    elif pred_prob > 0.4: risk="Medium"
    else: risk="Low"

    print(f"\nPredicted Churn: {'Yes' if pred==1 else 'No'}")
    print(f"Churn Probability: {pred_prob*100:.2f}%")
    print(f"Risk Category: {risk}")

    # Business suggestion
    if risk=="High":
        print("Suggestion: Offer retention incentives or loyalty programs.")
    elif risk=="Medium":
        print("Suggestion: Engage customer with reminders or offers.")
    else:
        print("Suggestion: Maintain current service.")

# ====== TEST UI ======
# predict_churn()  # Uncomment to run in terminal


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

# ====== ROC-AUC CURVE ======
# Get predicted probabilities for the positive class
y_pred_proba = cb_model.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color="blue", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="red", linestyle="--")  # random chance line
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()

# ====== CONFUSION MATRIX ======
y_pred = cb_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred, labels=cb_model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cb_model.classes_)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

# ====== PRINT ROC-AUC SCORE ======
print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_proba))

In [ ]:
predict_churn()


--- Enter Customer Details ---
Enter valid option: ['Male', 'Female']


In [ ]:
import pandas as pd
import pickle
from catboost import CatBoostClassifier, Pool

# Load dataset
df = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Clean TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Add tenure group (same logic you used in UI)
def tenure_to_group(tenure):
    tenure = float(tenure)
    if tenure <= 12: return '1'
    elif tenure <= 24: return '2'
    elif tenure <= 48: return '3'
    elif tenure <= 60: return '4'
    else: return '5'
df['TenureGroup'] = df['tenure'].apply(tenure_to_group)

# Replace 'No internet service' or 'No phone service' with 'No' for relevant columns
replace_cols = ['MultipleLines','OnlineSecurity','OnlineBackup','DeviceProtection',
                'TechSupport','StreamingTV','StreamingMovies']
for col in replace_cols:
    df[col] = df[col].replace({'No internet service':'No','No phone service':'No'})

# Binary mapping for relevant columns
binary_cols = ['Partner','Dependents','PhoneService','PaperlessBilling','Churn']
for col in binary_cols:
    df[col] = df[col].map({'Yes':1,'No':0})

# Compute NumServices
def compute_num_services(row):
    count = 0
    count += 1 if row['PhoneService'] == 1 else 0
    for k in ['MultipleLines','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']:
        count += 1 if row[k] == 'Yes' else 0
    return count
df['NumServices'] = df.apply(compute_num_services, axis=1).astype(int) # Ensure integer type

# Target
y = df['Churn']

# Features
X = df.drop(['customerID','Churn'], axis=1)

# Define categorical columns
cat_features = ['gender','MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
                'DeviceProtection','TechSupport','StreamingTV','StreamingMovies',
                'Contract','PaymentMethod','TenureGroup']

# Train CatBoost
model = CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1, loss_function='Logloss', verbose=50)
model.fit(X, y, cat_features=cat_features)

# Save model
with open("your_file.pkl", "wb") as f:
    pickle.dump(model, f)

print("✅ Model trained and saved as your_file.pkl")

In [ ]:
import gradio as gr
import pandas as pd
import pickle
import os
from catboost import Pool

# ---------- Load Model ----------
MODEL_PATH = "your_file.pkl"
cat_features = ['gender', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
                'Contract', 'PaymentMethod', 'TenureGroup']

model = None
if os.path.exists(MODEL_PATH):
    with open(MODEL_PATH, "rb") as f:
        model = pickle.load(f)

# ---------- Helper Functions ----------
def safe_total_charges(x):
    try:
        return float(str(x).strip().replace(" ", "0"))
    except:
        return 0.0

def compute_num_services(d):
    count = 0
    count += 1 if d['PhoneService'] == 1 else 0
    for k in ['MultipleLines','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']:
        count += 1 if d[k] == 'Yes' else 0
    return count

def tenure_to_group(tenure):
    tenure = float(tenure)
    if tenure <= 12: return '1'
    elif tenure <= 24: return '2'
    elif tenure <= 48: return '3'
    elif tenure <= 60: return '4'
    else: return '5'

# ---------- Prediction ----------
def predict_churn(gender, SeniorCitizen, Partner, Dependents, tenure, PhoneService, MultipleLines,
                  InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport,
                  StreamingTV, StreamingMovies, Contract, PaperlessBilling, PaymentMethod,
                  MonthlyCharges, TotalCharges):
    if model is None:
        return "⚠️ Model not loaded. Please train and save the model first."

    input_dict = {
        'gender': gender,
        'SeniorCitizen': 1 if SeniorCitizen == 'Yes' else 0,
        'Partner': 1 if Partner == 'Yes' else 0,
        'Dependents': 1 if Dependents == 'Yes' else 0,
        'tenure': tenure,
        'PhoneService': 1 if PhoneService == 'Yes' else 0,
        'MultipleLines': MultipleLines.replace('No phone service', 'No'),
        'InternetService': InternetService,
        'OnlineSecurity': OnlineSecurity.replace('No internet service', 'No'),
        'OnlineBackup': OnlineBackup.replace('No internet service', 'No'),
        'DeviceProtection': DeviceProtection.replace('No internet service', 'No'),
        'TechSupport': TechSupport.replace('No internet service', 'No'),
        'StreamingTV': StreamingTV.replace('No internet service', 'No'),
        'StreamingMovies': StreamingMovies.replace('No internet service', 'No'),
        'Contract': Contract,
        'PaperlessBilling': 1 if PaperlessBilling == 'Yes' else 0,
        'PaymentMethod': PaymentMethod,
        'MonthlyCharges': MonthlyCharges,
        'TotalCharges': safe_total_charges(TotalCharges),
    }
    input_dict['NumServices'] = compute_num_services(input_dict)
    input_dict['TenureGroup'] = tenure_to_group(tenure)

    input_df = pd.DataFrame([input_dict])
    pool = Pool(data=input_df, cat_features=cat_features)

    prob = model.predict_proba(pool)[:, 1][0]
    pred = model.predict(pool)[0]

    if prob > 0.7: risk = "🔥 High"
    elif prob > 0.4: risk = "🟡 Medium"
    else: risk = "🟢 Low"

    return f"""
### 🧮 Prediction Results
- **Predicted Churn:** {'✅ Yes' if pred==1 else '❌ No'}
- **Churn Probability:** {prob*100:.2f}%
- **Risk Category:** {risk}
"""

# ---------- Gradio UI ----------
with gr.Blocks(css="""
    body {background: linear-gradient(120deg, #f6d365, #fda085);}
    .gr-button {background: #ff7e5f !important; color: white !important; border-radius: 10px;}
    .gr-textbox textarea {background: #fff8e7;}
""") as demo:
    gr.Markdown("""
    <div style="text-align:center; padding:20px; background:linear-gradient(to right,#36d1dc,#5b86e5); border-radius:15px; color:white;">
        <h1>📊 Customer Churn Prediction</h1>
        <p>Enter customer details below to predict churn risk</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gender = gr.Dropdown(["Male","Female"], label="Gender")
            senior = gr.Dropdown(["Yes","No"], label="Senior Citizen")
            partner = gr.Dropdown(["Yes","No"], label="Partner")
            dep = gr.Dropdown(["Yes","No"], label="Dependents")
            tenure = gr.Number(label="Tenure (months)")
            phone = gr.Dropdown(["Yes","No"], label="Phone Service")
            multi = gr.Dropdown(["Yes","No","No phone service"], label="Multiple Lines")
            internet = gr.Dropdown(["DSL","Fiber optic","No"], label="Internet Service")
            onsec = gr.Dropdown(["Yes","No","No internet service"], label="Online Security")
            onb = gr.Dropdown(["Yes","No","No internet service"], label="Online Backup")
        with gr.Column(scale=1):
            dev = gr.Dropdown(["Yes","No","No internet service"], label="Device Protection")
            tech = gr.Dropdown(["Yes","No","No internet service"], label="Tech Support")
            stv = gr.Dropdown(["Yes","No","No internet service"], label="Streaming TV")
            sm = gr.Dropdown(["Yes","No","No internet service"], label="Streaming Movies")
            contract = gr.Dropdown(["Month-to-month","One year","Two year"], label="Contract Type")
            paper = gr.Dropdown(["Yes","No"], label="Paperless Billing")
            pay = gr.Dropdown(["Electronic check","Mailed check","Bank transfer (automatic)","Credit card (automatic)"], label="Payment Method")
            monthly = gr.Number(label="Monthly Charges")
            total = gr.Number(label="Total Charges")

    submit = gr.Button("🚀 Predict Churn", variant="primary")
    output = gr.Markdown()

    submit.click(
        fn=predict_churn,
        inputs=[gender, senior, partner, dep, tenure, phone, multi, internet, onsec, onb,
                dev, tech, stv, sm, contract, paper, pay, monthly, total],
        outputs=output
    )

demo.launch()
